# agri-drone — Notebook 1: full experimental matrix

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashut0sh-mishra/agri-drone/blob/research-upgrade/notebooks/colab/01_run_matrix.ipynb)

Runs the research-upgrade evaluation matrix end-to-end on Colab.
- **Quick mode** (default, T4, ~2 h) = 8 cells
- **Full mode** (A100, ~12 h) = 2400 cells

Outputs land under `MyDrive/agri-drone/results_v2/`.


## 1. Mount Drive + clone repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!test -d agri-drone || git clone https://github.com/Ashut0sh-mishra/agri-drone.git
%cd /content/agri-drone
!git fetch --all && git checkout main && git reset --hard origin/main
!pip install -q -r requirements.txt


## 2. GPU sanity check


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU.'
print('CUDA', torch.version.cuda, 'device:', torch.cuda.get_device_name(0))


## 3. Data source

Pre-create in your Drive and upload datasets:
```
MyDrive/agri-drone/data/plantvillage/
MyDrive/agri-drone/data/PDT_datasets/
```
See `docs/data_availability.md` for URLs and licences.


In [ ]:
#@title Data source
DATA_SOURCE = 'gdrive' #@param ['gdrive','kaggle','zenodo']
DRIVE_DATA = '/content/drive/MyDrive/agri-drone/data'
DRIVE_OUT  = '/content/drive/MyDrive/agri-drone/results_v2'
import os, pathlib
pathlib.Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
if DATA_SOURCE == 'gdrive':
    assert os.path.isdir(DRIVE_DATA), f'Missing {DRIVE_DATA}. Upload dataset first.'
    !mkdir -p datasets/externals && ln -sfn "$DRIVE_DATA" datasets/externals/drive
elif DATA_SOURCE == 'kaggle':
    print('Configure ~/.kaggle/kaggle.json and pull per scripts/download_data.py')
elif DATA_SOURCE == 'zenodo':
    print('Replace with your Zenodo archive URL and unzip to datasets/externals/')
print('(SHA256 validation advisory per docs/data_availability.md)')


## 4. Mode selector


In [ ]:
#@title Mode
#@markdown Pick one of: `quick` (T4, ~2h, 8 cells), `full` (A100, ~12h, 2400 cells),
#@markdown `large` (A100/T4, ~24-48h, 54 cells across PlantVillage + PlantDoc + iNaturalist-plants).
MODE = 'quick' #@param ['quick', 'full', 'large']
MATRIX_CONFIG = f'configs/matrix/{MODE}.yaml'
if MODE == 'full':
    print('*** FULL MODE: 2400 cells, ~12 h on A100. Need Colab Pro. ***')
elif MODE == 'large':
    print('*** LARGE MODE: 54 cells on PV + PlantDoc + iNat. A100 strongly recommended. ***')
    print('*** HF_TOKEN will be auto-loaded from Colab Secrets. ***')
    import os
    if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
        try:
            from google.colab import userdata
            os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
            print('Loaded HF_TOKEN from Colab secrets.')
        except Exception as e:
            print('WARNING: HF_TOKEN not found in env or Colab secrets ->', e)
    # Ensure `datasets` is available even if cell 1 ran on a cached env.
    try:
        import datasets  # noqa: F401
    except Exception:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=2.14.0'])
else:
    print('QUICK MODE: 8 cells, ~2 h on T4.')
print('config =', MATRIX_CONFIG)


## 5. Run the matrix


In [ ]:
#@title Run the matrix (auto-resumes, auto-retries on transient failures)
# This cell is safe to re-run after a disconnect: the runner appends to
# per_run.jsonl and skips cells that already finished with status=ok.
import json, os, subprocess, sys, time, pathlib

run_id = pathlib.Path(MATRIX_CONFIG).stem  # 'quick' | 'full' | 'large'
# run_id in the yaml may differ (e.g. 'quick-colab-t4'); parse it once.
try:
    import yaml
    _cfg = yaml.safe_load(open(MATRIX_CONFIG, 'r', encoding='utf-8'))
    run_id = _cfg.get('run_id', run_id)
except Exception:
    pass

per_run = pathlib.Path(DRIVE_OUT) / 'matrix' / run_id / 'per_run.jsonl'
print('per_run path:', per_run)
print('config:', MATRIX_CONFIG, 'exists:', pathlib.Path(MATRIX_CONFIG).exists())

def _count_ok():
    if not per_run.exists():
        return 0
    n = 0
    try:
        for ln in per_run.read_text(encoding='utf-8').splitlines():
            if not ln.strip():
                continue
            try:
                if json.loads(ln).get('status') == 'ok':
                    n += 1
            except Exception:
                pass
    except Exception:
        pass
    return n

MAX_OUTER_ATTEMPTS = 6  # survives up to 6 crashes/disconnects
for outer in range(1, MAX_OUTER_ATTEMPTS + 1):
    before = _count_ok()
    print(f'\n===== OUTER ATTEMPT {outer}/{MAX_OUTER_ATTEMPTS}  already_ok={before} =====')
    # Stream child output line-by-line so we see every print immediately and
    # never silently swallow a crash.
    cmd = [sys.executable, '-u', 'evaluate/matrix/run_matrix.py',
           '--config', MATRIX_CONFIG,
           '--out-dir', str(pathlib.Path(DRIVE_OUT) / 'matrix'),
           '--tracker', 'none']
    print('  cmd:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='')
    finally:
        proc.stdout.close()
    rc = proc.wait()
    after = _count_ok()
    print(f'\n  exit={rc}  ok_now={after}  progress={after - before}')
    if rc == 0:
        print('  runner exited cleanly -> done.')
        break
    if after == before:
        print('  no forward progress this pass; sleeping 30 s then retrying...')
        time.sleep(30)
    else:
        print('  partial progress saved; retrying remaining cells...')
        time.sleep(5)
else:
    print('!! hit MAX_OUTER_ATTEMPTS. Re-run this cell to keep going.')

print('\nFinal ok count:', _count_ok())
print('Results at:', per_run)


## 6. Post-matrix analysis (stats v2, EML, baseline audit, PDT sweep)


## 7. Auto-generate RESULTS_SUMMARY.md


In [ ]:
import json, pathlib
out = pathlib.Path(DRIVE_OUT); out.mkdir(parents=True, exist_ok=True)
def _load(rel):
    p = pathlib.Path('evaluate/results/v2') / rel
    return json.loads(p.read_text()) if p.exists() else {}
eml = _load('eml/headline_v4.json')
pdt = _load('pdt/threshold_sweep.json')
lines = [
    '# Results summary (auto-generated by notebook 1)',
    f'- Matrix config: `{MATRIX_CONFIG}`',
    f'- EML headline (INR/ha): {eml.get("headline_eml_inr_per_ha","[TO BE RE-RUN]")}',
    f'- EML sensitivity scenario: {eml.get("sensitivity_scenario_eml_inr_per_ha","[TO BE RE-RUN]")}',
    f'- PDT ROC-AUC: {pdt.get("roc_auc","[TO BE RE-RUN]")}',
    f'- PDT spec@90%sens: {pdt.get("specificity_at_90_sensitivity","[TO BE RE-RUN]")}',
]
(out / 'RESULTS_SUMMARY.md').write_text('\n'.join(lines))
print((out / 'RESULTS_SUMMARY.md').read_text())


## 8. Zip artifacts + print PR-comment command


In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M%S')
archive = f'/content/drive/MyDrive/agri-drone/results_v2_{stamp}'
shutil.make_archive(archive, 'zip', DRIVE_OUT)
print('Saved:', archive + '.zip')
print('\nPost summary back to PR locally with:')
print('  gh pr comment research-upgrade --body-file RESULTS_SUMMARY.md')
